# 00 — Environment, Learning Map, and the ANN Contract (TensorFlow / Keras)

This notebook establishes the **technical contract** used by every notebook in this branch. The goal is not to memorize APIs. The goal is to understand what an artificial neural network computes, why training changes its parameters, how hidden representations emerge, and how those mechanics become a reliable software system.

## What you will be able to explain by the end

1. How a 28×28 MNIST image becomes a 784-dimensional numerical vector.
2. Why a neuron computes an affine transformation followed by a nonlinearity.
3. How a full forward pass turns pixels into class evidence.
4. Why softmax and cross-entropy are paired for multiclass classification.
5. How gradients encode **parameter sensitivity** rather than vague "importance".
6. How backpropagation applies the chain rule through the computational graph.
7. What the optimizer actually changes after `backward()` / `GradientTape`.
8. How hidden representations move during training.
9. How to evaluate, debug, serialize, reload, and monitor a trained ANN.
10. How model metrics connect to operational and business decisions.

## Shared architecture

The primary classifier is:

$$
784 \rightarrow 64 \rightarrow 10
$$

with a nonlinear hidden layer and ten output logits. A separate 2D bottleneck model is used later to make representation learning visible.

## Reproducibility contract

- Python 3.11
- fixed random seed
- official MNIST archive
- deterministic balanced teaching subset: 5,000 train / 1,000 test
- committed notebook outputs
- CI re-executes every notebook from cleared state
- zero tolerated execution errors


In [1]:
from pathlib import Path
import urllib.request
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)
MNIST_PATH = DATA_DIR / 'mnist.npz'
MNIST_URL = 'https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz'
if not MNIST_PATH.exists():
    print('Downloading official MNIST archive...')
    urllib.request.urlretrieve(MNIST_URL, MNIST_PATH)
with np.load(MNIST_PATH) as data:
    x_train_raw, y_train_raw = data['x_train'], data['y_train']
    x_test_raw, y_test_raw = data['x_test'], data['y_test']
print('raw train:', x_train_raw.shape, y_train_raw.shape)
print('raw test :', x_test_raw.shape, y_test_raw.shape)


raw train: (60000, 28, 28) (60000,)
raw test : (10000, 28, 28) (10000,)


In [2]:
print('Python learning contract is active.')
print('MNIST classes:', np.unique(y_train_raw))
print('Each image contains', x_train_raw.shape[1] * x_train_raw.shape[2], 'pixels.')


Python learning contract is active.
MNIST classes: [0 1 2 3 4 5 6 7 8 9]
Each image contains 784 pixels.


## Business lens

MNIST itself is not a business problem; it is a controlled laboratory. The same pipeline appears in real systems: image inspection, document classification, fraud screening, product categorization, intent prediction, and quality control. The educational value comes from being able to inspect the entire causal chain on a dataset simple enough to visualize.
